# Machine Learning: Regressions

**Objectives:**
- Load and explore a dataset
- Split data into training and test sets
- Train regression models with scikit-learn (Linear, Lasso, Ridge)
- Evaluate model performance using $R^2$ scores
- Understand the purpose of cross-validation (k-fold)

We work through two datasets:  
1. **Diabetes dataset** — a small, pre-normalized dataset to learn the ML workflow  
2. **California Housing dataset** — a larger, real-world dataset to practice sparse regressions

---
## Part 1: Diabetes Dataset — Basic Regression

### Step 1: Import and Explore the Data

We start by loading the **diabetes dataset** from scikit-learn.  
It contains 10 baseline variables (age, sex, BMI, blood pressure, etc.) for 442 patients,  
and a target variable measuring disease progression one year later.

The dataset is returned as a dictionary with keys: `'data'`, `'target'`, `'feature_names'`, `'DESCR'`.

In [ ]:
import sklearn
import sklearn.datasets

dataset = sklearn.datasets.load_diabetes()

# The result is a dictionary:
# 'data'          -> features (X)
# 'target'        -> labels (y)
# 'feature_names' -> names of the features
# 'DESCR'         -> description of the dataset

In [ ]:
print(dataset['DESCR'])

Let's put the data into a **pandas DataFrame** for easier exploration.

In [ ]:
import pandas

df = pandas.DataFrame(dataset['data'], columns=dataset['feature_names'])
df['disease_progression'] = dataset['target']

In [ ]:
df.describe()

> **Interpretation:** Notice that the means of the features are close to zero and the standard deviations are similar across variables. This tells us the data has already been **normalized** (centered and scaled). Normalization is important so that no single feature dominates the regression just because of its scale.

In [ ]:
import seaborn
seaborn.pairplot(df)

---
### Step 2: Split Into Training and Test Sets

**Why do we split?** We want to evaluate our model on data it has *never seen* during training.  
This tells us how well the model **generalizes** to new observations (out-of-sample performance).

We use 70% for training and 30% for testing.

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
# Features: one row per observation, one column per feature
print("Features shape:", dataset['data'].shape)

# Target: what we are trying to predict (disease progression)
print("Target shape:", dataset['target'].shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    dataset['data'], dataset['target'],
    test_size=0.3,
    random_state=56  # fixed seed for reproducibility
)

---
### Step 3: Train a Linear Regression Model

**What is linear regression?**  
We fit a model: $\hat{y} = a + b_1 x_1 + b_2 x_2 + \ldots + b_{10} x_{10}$  
where $a$ is the intercept and $b_i$ are the coefficients.

Scikit-learn makes this very simple: create a model object, then call `.fit()`.

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()  # create model object
model.fit(X_train, y_train)  # train it on the training data

In [ ]:
# Intercept (a)
print("Intercept:", model.intercept_)

In [ ]:
# Coefficients (b_1, b_2, ..., b_10)
print("Coefficients:", model.coef_)

---
### Step 4: Evaluate the Model

We use the **$R^2$ score** to evaluate model quality.  
$R^2 = 1$ means perfect prediction; $R^2 = 0$ means the model predicts no better than the mean.

We check *both* the training score and the test score to look for **overfitting**.  
If the training score is much higher than the test score, the model memorized the training data rather than learning general patterns.

In [ ]:
print("Test R² score: ", model.score(X_test, y_test))
print("Train R² score:", model.score(X_train, y_train))

> **Interpretation:** The test and training scores are relatively close, so overfitting is not a major concern here. However, the $R^2$ is moderate (~0.5), meaning the linear model captures only about half of the variance in disease progression.

---
### Step 5: Should We Adjust the Test Set Size?

One might wonder: *Does the size of the test set matter?*  
Let's try different splits and see how the score changes.

> ⚠️ **Warning:** This is shown for illustration only — it is **bad practice** because we are repeatedly peeking at the test data to make decisions.  

In [ ]:
sizes = [0.01, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
scores = []

for s in sizes:
    X_train, X_test, y_train, y_test = train_test_split(
        dataset['data'], dataset['target'], test_size=s
    )
    model = LinearRegression()
    model.fit(X_train, y_train)
    score = model.score(X_test, y_test)
    scores.append(score)

In [ ]:
from matplotlib import pyplot as plt
plt.plot(sizes, scores, marker='o')
plt.xlabel('Test set fraction')
plt.ylabel('R² score')
plt.title('Score vs. test set size')
plt.show()

> **Interpretation:** The score fluctuates depending on the split. This illustrates a key problem: a *single* train/test split gives an unstable estimate of model performance. The solution? **Cross-validation.**

---
### Step 6: K-Fold Cross-Validation

**Why cross-validation?** Instead of relying on one random split, we rotate which part of the data is used for testing.  
With $k$-fold, the data is split into $k$ equal parts. Each part takes a turn as the test set while the other $k-1$ parts are used for training. We then average the $k$ scores for a more stable estimate.

In [ ]:
from sklearn.model_selection import KFold

X = dataset['data']
y = dataset['target']
scores = []

kf = KFold(n_splits=3)

for train_index, test_index in kf.split(X):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    model = LinearRegression()
    model.fit(X_train, y_train)
    score = model.score(X_test, y_test)
    scores.append(score)

print("K-Fold scores:", scores)
print("Mean score:   ", sum(scores) / len(scores))

> **Interpretation:** The k-fold scores give us a more reliable estimate of the model's predictive power. The average across folds is a better summary than any single split.

---
### Step 7: Introduction to Lasso Regression

**Lasso** (Least Absolute Shrinkage and Selection Operator) adds a penalty to the size of the coefficients:  
it pushes some coefficients toward zero, effectively performing **variable selection**.

This can be useful when you suspect that not all features are relevant.

In [ ]:
from sklearn.linear_model import Lasso

X_train, X_test, y_train, y_test = train_test_split(
    dataset['data'], dataset['target'], test_size=0.3, random_state=56
)

model_lasso = Lasso()
model_lasso.fit(X_train, y_train)
print("Lasso test R²:", model_lasso.score(X_test, y_test))

> **Interpretation:** The Lasso score on the test set is slightly worse than plain linear regression. This is because the **default regularization parameter** (`alpha=1.0`) may be too strong for this dataset, shrinking useful coefficients too much. Tuning `alpha` could improve performance, but that is beyond the scope of this exercise.

---
## Part 2: California Housing — Sparse Regressions

Now we apply the same workflow to a larger, real-world dataset:  
the **California Housing** dataset (median house values across California districts).

### Step 1: Load and Explore the Data

In [ ]:
from sklearn.datasets import fetch_california_housing
dataset = fetch_california_housing()

In [ ]:
print(dataset['DESCR'])

### Step 2: Split the Data

Same approach: 70% training, 30% test.

In [ ]:
X = dataset['data']
y = dataset['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=58
)

### Step 3: Lasso Regression

> **Note:** Ideally, we should check whether the data needs normalization before fitting regularized models. For this exercise we proceed with default settings.

In [ ]:
from sklearn.linear_model import Lasso

model_lasso = Lasso()
model_lasso.fit(X_train, y_train)
print("Lasso R² on test set:", model_lasso.score(X_test, y_test))

### Step 4: Ridge Regression

**Ridge** regression also penalizes large coefficients, but uses an $L_2$ penalty (squared coefficients) instead of $L_1$ (absolute values). Ridge keeps all features in the model but shrinks coefficients.

In [ ]:
from sklearn.linear_model import Ridge

model_ridge = Ridge()
model_ridge.fit(X_train, y_train)
print("Ridge R² on test set:", model_ridge.score(X_test, y_test))

> **Interpretation:** The Ridge model achieves a better $R^2$ than Lasso on this dataset. This suggests that most features contribute to predicting house prices — Ridge's gentler shrinkage is more appropriate here than Lasso's aggressive variable selection.

> ⚠️ **Important caveat:** We used the *same* test set to compare Lasso and Ridge. Strictly speaking, this means the test set influenced our model choice. A cleaner approach would be to hold out a *separate* validation set (or use cross-validation) to choose the model, and only evaluate the final model on the test set once.